# Séance de révisions

Dans ce notebook, vous trouverez une séance d'application de FastText, où nous allons approfondir cette fois à deux auteurs (Flaubert et Zola), afin de détecter les variations en vocabulaire de ces deux textes.
**À la différences des autres TP, celui-ci doit être réalisé avec une plus grande autonomie**.

## Obtention des textes

In [2]:
def clean_up_text(texte_brut):
    marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
    marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"

    texte = texte_brut.split(marque_debut, 1)[1]
    texte = texte.split(marque_fin, 1)[0].strip()

    return texte

In [4]:
## Téléchargements des livres

from pathlib import Path
from urllib.request import Request, urlopen

URLS = [
    "https://www.gutenberg.org/cache/epub/14155/pg14155.txt",
    "https://www.gutenberg.org/cache/epub/48881/pg48881.txt",
    "https://www.gutenberg.org/cache/epub/56528/pg56528.txt",
    "https://www.gutenberg.org/cache/epub/8600/pg8600.txt"
]

textes = []

for url in URLS:
    fichier = Path(f"corpus/{url.split("/")[-1]}")

    if not fichier.exists():
        fichier.parent.mkdir(parents=True, exist_ok=True)
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse:
            contenu = reponse.read().decode("utf-8-sig")
        fichier.write_text(clean_up_text(contenu), encoding="utf-8")

    texte_brut = fichier.read_text(encoding="utf-8")
    print(f"{len(texte_brut):,} caractères chargés")

703,078 caractères chargés
695,334 caractères chargés
1,019,631 caractères chargés
890,204 caractères chargés


In [25]:
## Téléchargement des modèles FastText

import gzip
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.bin.gz"
fichier_gz = Path("modele/cc.fr.300.bin.gz")
fichier_bin = Path("modele/cc.fr.300.bin")

if not fichier_bin.exists():
    fichier_gz.parent.mkdir(parents=True, exist_ok=True)
    if not fichier_gz.exists():
        print("téléchargement de", url)
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse, fichier_gz.open("wb") as sortie:
            shutil.copyfileobj(reponse, sortie)
    with gzip.open(fichier_gz, "rb") as entree, fichier_bin.open("wb") as sortie:
        shutil.copyfileobj(entree, sortie)

print("fichier prêt :", fichier_bin)

fichier prêt : modele/cc.fr.300.bin


In [29]:
## Téléchargement des stop words en français
stop_words_url = "https://raw.githubusercontent.com/stopwords-iso/stopwords-fr/refs/heads/master/stopwords-fr.txt"
response = Request(stop_words_url, headers={"User-Agent": "Mozilla/5.0"})
STOP_WORDS = urlopen(response).read().decode("utf-8").split("\n")

## Exercice


1. Trouvez pour chaque roman les mots les plus fréquemment utilisés.
2. Refaîtes l'exercice en enlevant cette fois les mots courants, tels que définis dans la variable `STOP_WORDS`.
3. À l'aide de la bibliothèqe `gensim` et en utilisant les vecteurs `FastText` téléchargés précedemment, réalisez une projection de chacun des romans, en ayant au préalable enlevé les `STOP_WORDS`.
4. Appliquez une SVD en dimension 2 sur les vecteurs Word2Vec et proposez une visualisation de l'ensemble des mots, en colorant chaque mot en fonction de son roman d'origine. Commentez le résultat.
5. Représentez chaque roman comme un vecteur unique en calculant la moyenne des vecteurs des mots qui le compose.
6. Calculez la similarité cosinus entre chacun les 4 documents. Que pouvez-vous en conclure?